# Phase 1 - Validation Suite: State Nonfarm Payrolls (CES)

**Scope:** validate the state total-nonfarm-payroll dataset (`{ABBR}NA`, monthly seasonally adjusted, level in thousands, BLS Current Employment Statistics via FRED/ALFRED) before any modeling. NSA (`{ABBR}NAN`) is pulled for the seasonality contrast; national PAYEMS for reconciliation.

**Downstream uses and what they stress:**

| Use case | Failure modes probed here |
|---|---|
| Forecasting | trend/stationarity, residual seasonality in SA data, outliers, breaks (V5, V7) |
| Lead-lag | timestamp & alignment conventions, sign orientation vs unemployment (V3, V8) |
| Backtesting | point-in-time integrity, **benchmark revisions**, publication lag, look-ahead guards (V3, V4, V9) |

**How NFP differs from continued claims (the sister dataset).** These two series sit at opposite ends of the revision spectrum, and the suite is tuned accordingly:

| | State continued claims | **State nonfarm payrolls** |
|---|---|---|
| Frequency | weekly (Sat week-ending) | **monthly** (pay period incl. the 12th) |
| Seasonal adjustment | NSA only | **SA** (seasonality should be *absent* - a QC flip in V7) |
| Revised after first print? | ~0% (essentially never) | **~99%** - monthly re-estimates + **annual benchmark** revisions to QCEW |
| Level behaviour | cyclical, mean-reverting | **trending** (I(1) with drift) |
| Publication lag | ~10 days | **~50 days** (joint State Employment & Unemployment report, ~3 wks after the national jobs report) |
| Sum-of-states vs national | PR/VI gap (~1.4%) | **near-exact** (ratio ~1.0002) |

The headline consequence: for claims, current data ≈ first release, so point-in-time was a *nicety*. For NFP, **PIT is mandatory** - current values are contaminated by future benchmark revisions the modeller could not have known (V4, V9). Each check states its question and pass criterion up front; the last cells render the report and go/no-go gates.

In [ ]:
import os, time, json, hashlib
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FRED_API_KEY = os.getenv("FRED_API_KEY", "12d77a40907e43a92e9a295801db18d2")
FRED_URL     = "https://api.stlouisfed.org/fred/series/observations"

DATA_FILES = {
    "vintage": "nfp_vintage_first_release.csv",
    "current": "nfp_current_history.csv",
    "pit":     "nfp_pit_features.csv",
}

# ── Check registry ────────────────────────────────────────────────────────────
checks = []

def add_check(module, check_id, name, result, metric="", threshold="", note=""):
    assert result in ("PASS", "WARN", "FAIL")
    checks.append(dict(module=module, id=check_id, check=name, result=result,
                       metric=str(metric), threshold=str(threshold), note=note))
    print(f"[{result:4s}] {check_id:6s} {name}  |  {metric}  (criterion: {threshold})")
    if note:
        print(f"       note: {note}")


def fred_get(params, retries=5):
    params = {**params, "api_key": FRED_API_KEY, "file_type": "json"}
    for attempt in range(retries):
        data = requests.get(FRED_URL, params=params, timeout=180).json()
        if "error_message" in data:
            if "Too Many Requests" in data["error_message"]:
                time.sleep(2 ** (attempt + 1)); continue
            raise ValueError(data["error_message"])
        return data
    raise RuntimeError(f"Rate limit persists: {params.get('series_id')}")


def fred_obs(params, retries=5):
    return [o for o in fred_get(params, retries).get("observations", []) if o["value"] != "."]

print("Setup ok.")

## Maintained hypotheses - CES publication mechanics

**H1 - Publication schedule.** State total nonfarm payrolls are published in the BLS *State Employment and Unemployment* report, released monthly roughly **three weeks after** the national Employment Situation (jobs) report - about **7 weeks (~50 days) after the reference month**, typically on a Friday.
→ tested by **V3.2** (release weekday), **V3.3** (publication lag ≈ 50 days).

**H2 - Reference period.** The payroll figure counts employees on payrolls for the pay period that **includes the 12th** of the reference month; FRED stamps the observation on the **1st of that month**.
→ tested by **V2.1** (all observation dates are month-starts).

**H3 - Revision structure.** CES estimates are revised in the two months after first release (sample completion) and then **benchmarked annually** to the near-universe QCEW count - the benchmark, published each winter with the January data, restates roughly the prior 21 months. This makes current data a poor proxy for what was known in real time.
→ quantified by **V4** (first-vs-current revision, benchmark-month clustering) and **V9** (feature contamination).

**Vocabulary.** *First release* = the initial (preliminary) estimate for a month, stamped by its `first_release_date`. *Current* = the latest value after all monthly and benchmark revisions. *Benchmark revision* = the annual QCEW re-anchoring. *Two clocks* = observation time (the month measured, V2) vs information time (when it became known, V3).

## V0 - Build the dataset from the FRED/ALFRED API (self-contained)

Fetches everything it validates - no other notebook needs to run first. **V0a** pulls, per state, the SA ALFRED first-release table (`output_type=4`), the SA current history and the NSA current history (~155 calls, ~3-4 min). **V0b** assembles the monthly point-in-time panel, persists three CSVs and pulls national benchmarks. Set `USE_CACHE = True` to reuse a previous run's CSVs.

In [ ]:
# ── V0a. Fetch state NFP data from the FRED/ALFRED API ────────────────────────
USE_CACHE = False   # True -> reuse CSVs from a previous run (skips the ~3-4 min fetch)

STATES = [
    ("Alabama","AL","01"),("Alaska","AK","02"),("Arizona","AZ","04"),("Arkansas","AR","05"),
    ("California","CA","06"),("Colorado","CO","08"),("Connecticut","CT","09"),("Delaware","DE","10"),
    ("District of Columbia","DC","11"),("Florida","FL","12"),("Georgia","GA","13"),("Hawaii","HI","15"),
    ("Idaho","ID","16"),("Illinois","IL","17"),("Indiana","IN","18"),("Iowa","IA","19"),
    ("Kansas","KS","20"),("Kentucky","KY","21"),("Louisiana","LA","22"),("Maine","ME","23"),
    ("Maryland","MD","24"),("Massachusetts","MA","25"),("Michigan","MI","26"),("Minnesota","MN","27"),
    ("Mississippi","MS","28"),("Missouri","MO","29"),("Montana","MT","30"),("Nebraska","NE","31"),
    ("Nevada","NV","32"),("New Hampshire","NH","33"),("New Jersey","NJ","34"),("New Mexico","NM","35"),
    ("New York","NY","36"),("North Carolina","NC","37"),("North Dakota","ND","38"),("Ohio","OH","39"),
    ("Oklahoma","OK","40"),("Oregon","OR","41"),("Pennsylvania","PA","42"),("Rhode Island","RI","44"),
    ("South Carolina","SC","45"),("South Dakota","SD","46"),("Tennessee","TN","47"),("Texas","TX","48"),
    ("Utah","UT","49"),("Vermont","VT","50"),("Virginia","VA","51"),("Washington","WA","53"),
    ("West Virginia","WV","54"),("Wisconsin","WI","55"),("Wyoming","WY","56"),
]
# Series: {ABBR}NA = SA total nonfarm (validated series); {ABBR}NAN = NSA (seasonality contrast)


def fetch_vintage(series_id):
    return fred_obs({"series_id": series_id, "realtime_start": "1776-07-04",
                     "realtime_end": "9999-12-31", "output_type": 4, "limit": 100000})

def fetch_current(series_id):
    return fred_obs({"series_id": series_id, "observation_start": "1990-01-01", "limit": 100000})


if USE_CACHE and all(os.path.exists(f) for f in DATA_FILES.values()):
    df_v = pd.read_csv(DATA_FILES["vintage"], parse_dates=["obs_date", "first_release_date"])
    df_c = pd.read_csv(DATA_FILES["current"], parse_dates=["obs_date"])
    pit  = pd.read_csv(DATA_FILES["pit"], parse_dates=["as_of_date", "nfp_latest_obs"])
    print("Loaded cached CSVs (USE_CACHE=True).")
else:
    vintage_rows, current_rows, failed = [], [], []
    for i, (name, abbr, fips) in enumerate(STATES):
        try:
            for o in fetch_vintage(f"{abbr}NA"):
                vintage_rows.append({"state": name, "abbr": abbr, "fips": fips,
                    "obs_date": pd.to_datetime(o["date"]),
                    "first_release_date": pd.to_datetime(o["realtime_start"]),
                    "superseded_date": o["realtime_end"], "value_first_release": float(o["value"])})
            time.sleep(0.3)
            for o in fetch_current(f"{abbr}NA"):
                current_rows.append({"state": name, "abbr": abbr, "adjustment": "sa",
                    "obs_date": pd.to_datetime(o["date"]), "value_current": float(o["value"])})
            time.sleep(0.3)
            for o in fetch_current(f"{abbr}NAN"):
                current_rows.append({"state": name, "abbr": abbr, "adjustment": "nsa",
                    "obs_date": pd.to_datetime(o["date"]), "value_current": float(o["value"])})
            print(f"[{i+1:2d}/51] {abbr}NA / {abbr}NAN ok")
        except Exception as e:
            print(f"[{i+1:2d}/51] {abbr} FAILED - {e}"); failed.append(abbr)
        time.sleep(0.3)
    if failed:
        raise RuntimeError(f"Fetch failed for {failed} - re-run this cell (rate limits are transient).")
    df_v = pd.DataFrame(vintage_rows)
    df_v["is_current"]   = df_v["superseded_date"] == "9999-12-31"
    df_v["pub_lag_days"] = (df_v["first_release_date"] - df_v["obs_date"]).dt.days
    df_v = df_v.drop(columns="superseded_date")
    df_c = pd.DataFrame(current_rows)
    pit = None

print(f"vintage rows: {len(df_v):,}   current rows: {len(df_c):,}")

In [ ]:
# ── V0b. Assemble the monthly point-in-time panel, persist, fetch benchmarks ──
N_LAGS = 13   # months of history per as-of date (enables MoM, 3m, 6m, YoY)

if pit is None:
    AS_OF_GRID = pd.date_range(df_v["first_release_date"].min().to_period("M").to_timestamp(),
                               df_v["first_release_date"].max().to_period("M").to_timestamp(), freq="MS")

    def build_pit(vint, n_lags, grid):
        """Last n_lags monthly first-release readings available at each as-of month.
        Releases are append-only, so the cummax of release dates is the availability
        frontier; searchsorted gives the released-prefix length at each as-of date."""
        out = {}
        for state, g in vint.groupby("state", sort=False):
            g = g.sort_values("obs_date")
            rel_eff = g["first_release_date"].cummax().values
            vals = g["value_first_release"].values
            obsd = g["obs_date"].values
            n_avail = np.searchsorted(rel_eff, grid.values, side="right")
            for j, as_of in enumerate(grid):
                k = n_avail[j]
                if k < n_lags:
                    continue
                row = {f"nfp_t{m}": vals[k - 1 - m] for m in range(n_lags)}
                row["nfp_latest_obs"] = pd.Timestamp(obsd[k - 1])
                row["nfp_lag_months"] = (as_of.to_period("M") - pd.Timestamp(obsd[k - 1]).to_period("M")).n
                out[(as_of, state)] = row
        return out

    pit = pd.DataFrame([{"as_of_date": k[0], "state": k[1], **row}
                        for k, row in build_pit(df_v, N_LAGS, AS_OF_GRID).items()])
    pit["nfp_mom1_pct"] = (pit["nfp_t0"] / pit["nfp_t1"]  - 1) * 100
    pit["nfp_mom3_pct"] = (pit["nfp_t0"] / pit["nfp_t3"]  - 1) * 100
    pit["nfp_yoy_pct"]  = (pit["nfp_t0"] / pit["nfp_t12"] - 1) * 100
    df_v.to_csv(DATA_FILES["vintage"], index=False)
    df_c.to_csv(DATA_FILES["current"], index=False)
    pit.to_csv(DATA_FILES["pit"], index=False)
    print("Dataset written to CSVs.")

sa_c  = df_c[df_c["adjustment"] == "sa"].copy()    # SA current history (validated series)
nsa_c = df_c[df_c["adjustment"] == "nsa"].copy()   # NSA current history (seasonality contrast)
print(f"pit rows: {len(pit):,}   SA current: {len(sa_c):,}   NSA current: {len(nsa_c):,}")

# National benchmarks: PAYEMS (SA) current + first-release table + vintage dates
nat_cur = pd.DataFrame(fred_obs({"series_id": "PAYEMS", "observation_start": "1990-01-01", "limit": 100000}))
nat_cur["obs_date"] = pd.to_datetime(nat_cur["date"]); nat_cur["value"] = nat_cur["value"].astype(float)
nat_v = pd.DataFrame(fred_obs({"series_id": "PAYEMS", "realtime_start": "1776-07-04",
                               "realtime_end": "9999-12-31", "output_type": 4, "limit": 100000}))
nat_v["obs_date"] = pd.to_datetime(nat_v["date"]); nat_v["first_release_date"] = pd.to_datetime(nat_v["realtime_start"])
nat_v["value"] = nat_v["value"].astype(float)
nat_v["pub_lag_days"] = (nat_v["first_release_date"] - nat_v["obs_date"]).dt.days
nat_vdates = pd.to_datetime(pd.Series(requests.get(
    "https://api.stlouisfed.org/fred/series/vintagedates",
    params={"series_id": "PAYEMS", "api_key": FRED_API_KEY, "file_type": "json", "limit": 10000},
    timeout=60).json()["vintage_dates"]))
print(f"national PAYEMS: {len(nat_cur)} current obs, {len(nat_v)} first releases, {len(nat_vdates)} vintage dates")

## V1 - Structural integrity

| Check | Question it answers |
|---|---|
| V1.1 | Do we have all 51 states in every table? |
| V1.2 | Is (state, adjustment, month) a unique key? |
| V1.3 | Are key fields complete (no nulls)? |
| V1.4 | Is every payroll level strictly positive and finite? |
| V1.5 | Are the series lengths consistent across states? |

In [ ]:
n_states = {name: df["state"].nunique() for name, df in
            [("vintage", df_v), ("SA current", sa_c), ("pit", pit)]}
add_check("V1", "V1.1", "51 states present in all tables",
          "PASS" if all(v == 51 for v in n_states.values()) else "FAIL",
          metric=n_states, threshold="== 51 everywhere")

dup_v = df_v.duplicated(["state", "obs_date"]).sum()
dup_c = df_c.duplicated(["state", "adjustment", "obs_date"]).sum()
dup_p = pit.duplicated(["as_of_date", "state"]).sum()
add_check("V1", "V1.2", "No duplicate keys (vintage: state x month; current: state x adj x month)",
          "PASS" if dup_v + dup_c + dup_p == 0 else "FAIL",
          metric=f"vintage={dup_v}, current={dup_c}, pit={dup_p}", threshold="== 0")

nulls = int(df_v[["state", "obs_date", "first_release_date", "value_first_release"]].isna().sum().sum()
            + df_c[["state", "obs_date", "value_current"]].isna().sum().sum())
add_check("V1", "V1.3", "No nulls in key fields", "PASS" if nulls == 0 else "FAIL",
          metric=f"{nulls} nulls", threshold="== 0")

bad_v = int((~np.isfinite(df_v["value_first_release"])).sum() + (df_v["value_first_release"] <= 0).sum())
bad_c = int((~np.isfinite(df_c["value_current"])).sum() + (df_c["value_current"] <= 0).sum())
add_check("V1", "V1.4", "All payroll levels strictly positive and finite",
          "PASS" if bad_v + bad_c == 0 else "FAIL",
          metric=f"vintage bad={bad_v}, current bad={bad_c}", threshold="== 0",
          note="employment is a large positive stock (thousands) - no zeros/negatives expected, unlike claims")

rc = sa_c.groupby("state").size()
add_check("V1", "V1.5", "SA series length consistent across states",
          "PASS" if (rc.max() - rc.min()) == 0 else "WARN",
          metric=f"min={rc.min()}, max={rc.max()} months", threshold="identical",
          note="all SA state series begin 1990-01 by construction")

### V1 in depth: the shape of the data

Before trusting the *content*, map the *shape*: where observations are missing, how many states report each period, whether any (state, period) key is duplicated, and - for any zero values - whether they are ever revised to non-zero. Plot titles state whether they show **first-release** or **revised (current)** data.

In [ ]:
# ── V1 shape audit: missing-value map, coverage over time, duplicate keys ─────
# Operates on the primary REVISED (current) series. Reindexing the state x period panel to the full
# regular calendar exposes any missing observation as a NaN cell.
_prim = sa_c.copy()
_states_sorted = sorted(_prim["state"].unique())
_wide_all = (_prim.pivot(index="obs_date", columns="state", values="value_current")
             .reindex(columns=_states_sorted).sort_index())
_cal = pd.date_range(_wide_all.index.min(), _wide_all.index.max(), freq="MS")
_wide_all = _wide_all.reindex(_cal)
_missing = _wide_all.isna()

# (1) heatmap of MISSING observations: states on the x-axis, time on the y-axis
fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(_missing.values, aspect="auto", cmap="Greys", interpolation="nearest", vmin=0, vmax=1)
ax.set_xticks(range(len(_states_sorted))); ax.set_xticklabels(_states_sorted, rotation=90, fontsize=6)
_yt = np.linspace(0, len(_cal) - 1, 12).astype(int)
ax.set_yticks(_yt); ax.set_yticklabels([_cal[i].strftime("%Y-%m") for i in _yt], fontsize=7)
ax.set_xlabel("state"); ax.set_ylabel("period (time)")
ax.set_title("V1 - missing observations by state, REVISED (current) data  |  black = missing period, white = present")
plt.tight_layout(); plt.show()

# (2) counts of values reported per period over time (fields are never null per V1.3; this counts
#     how many of the 51 states have an observation each period)
_coverage = _wide_all.notna().sum(axis=1).rename("n_states_reporting").to_frame()
_coverage.index.name = "period"
print("V1 - values reported per period, REVISED (current) data (no missing fields; counts states present):\n")
print("distribution over all periods (how many periods have N states reporting):")
print(_coverage["n_states_reporting"].value_counts().sort_index().rename("n_periods").to_frame().T.to_string())
_incomplete = _coverage[_coverage["n_states_reporting"] < len(_states_sorted)]
print(f"\nincomplete periods (< {len(_states_sorted)} states): {len(_incomplete)} of {len(_cal)}")
if len(_incomplete):
    print(_incomplete.head(40).to_string())
else:
    print("  (complete panel - every period has all states)")

# (3) duplicate (state, period) keys
_dupmask = _prim.duplicated(["state", "obs_date"], keep=False)
print(f"\nV1 - duplicate (state, period) keys, REVISED (current) data: {int(_dupmask.sum())}")
if int(_dupmask.sum()):
    print(_prim[_dupmask].groupby(["state", "obs_date"]).size().rename("count").reset_index().to_string(index=False))
else:
    _cnt = _prim.groupby("state")["obs_date"].agg(n_obs="size", n_unique_periods="nunique")
    print("no duplicates - every (state, period) is unique. Per-state observation counts:")
    print(_cnt.describe().round(1).to_string())

In [ ]:
# ── V1.4/V1.4b - are zero (<=0) values ever revised to non-zero? ──────────────
# For every observation that is <= 0 in either the first-release or the revised (current) data, pull
# its full ALFRED trajectory (output_type=2) and lay out the value at each successive release date, so
# we can see whether a zero was later corrected upward. Each rel* column is 'release_date=value'.
# Where no vintage is archived (pre-archive obs), the first column falls back to the current value.
_prim = sa_c.copy()
_vint = df_v.copy()
_zero_cur = _prim.loc[_prim["value_current"] <= 0, ["state", "obs_date"]]
_zero_first = (_vint.loc[_vint["value_first_release"] <= 0, ["state", "obs_date"]]
               if "value_first_release" in _vint.columns else _prim.iloc[0:0][["state", "obs_date"]])
_zero_keys = pd.concat([_zero_cur, _zero_first]).drop_duplicates().sort_values(["state", "obs_date"])
_SID = {name: f"{ab}NA" for name, ab, fp in STATES}
_curmap = _prim.set_index(["state", "obs_date"])["value_current"]

_rows = []
for _st in sorted(_zero_keys["state"].unique()):
    try:
        _d = fred_get({"series_id": _SID[_st], "realtime_start": "1776-07-04", "realtime_end": "9999-12-31",
                       "output_type": 2, "limit": 100000})
        _t = pd.DataFrame(_d["observations"]).set_index("date")
        _vc = [c for c in _t.columns if c.startswith(_SID[_st])]
        _vals = _t[_vc].replace(".", np.nan).astype(float)
        _vals.columns = pd.to_datetime([c.split("_", 1)[1] for c in _vc])
    except Exception:
        _vals = None
    for _od in sorted(_zero_keys.loc[_zero_keys["state"] == _st, "obs_date"]):
        _key = pd.Timestamp(_od).strftime("%Y-%m-%d")
        _row = {"state": _st, "obs_date": _key}
        if _vals is not None and _key in _vals.index:
            _s = _vals.loc[_key].dropna()
            _steps = _s[_s != _s.shift()]                       # keep only where the value changed
            for _i, (_rd, _v) in enumerate(_steps.items(), start=1):
                _row[f"rel{_i}"] = f"{_rd.date()}={_v:g}"
            _row["ever_nonzero"] = bool((_s != 0).any())
        else:
            _cv = _curmap.get((_st, _od), np.nan)
            _row["rel1"] = f"current={_cv:g}"                    # vintage not archived -> current value
            _row["ever_nonzero"] = bool(pd.notna(_cv) and _cv > 0)
        _rows.append(_row)

_ztable = pd.DataFrame(_rows)
print("V1.4/V1.4b - zero-value revision trajectory (first release -> successive vintages):")
print("columns rel1, rel2, ... are 'release_date=value'; rel1 is the earliest archived release.\n")
if len(_ztable):
    print(_ztable.fillna("").to_string(index=False))
    print(f"\n{int(_ztable['ever_nonzero'].sum())} of {len(_ztable)} zero observations were non-zero in some "
          f"vintage (i.e. later revised away from zero).")
else:
    print("  (no zero / non-positive values in this dataset - nothing to track)")

## V2 - Calendar & frequency (observation clock)

Validates the **observation clock**: `obs_date` stamps the reference month (FRED uses the 1st; the payroll counts the pay period including the 12th). The information clock (when the number became known) is V3's subject.

| Check | Question it answers |
|---|---|
| V2.1 | Are all observation dates month-starts (monthly convention)? |
| V2.2 | Are there missing months in any state's history? |
| V2.3 | Do all states cover the same window? |

In [ ]:
non_month_start = int((df_c["obs_date"].dt.day != 1).sum())
add_check("V2", "V2.1", "All obs_date are month-starts (monthly reference convention)",
          "PASS" if non_month_start == 0 else "FAIL",
          metric=f"{non_month_start} non-first-of-month", threshold="== 0")

gaps = {}
for state, g in sa_c.groupby("state"):
    cal = pd.date_range(g["obs_date"].min(), g["obs_date"].max(), freq="MS")
    n = len(cal.difference(g["obs_date"]))
    if n:
        gaps[state] = n
add_check("V2", "V2.2", "No missing months per state (SA)",
          "PASS" if not gaps else "FAIL", metric=gaps or "0 gaps", threshold="== 0")

starts = sa_c.groupby("state")["obs_date"].min()
ends   = sa_c.groupby("state")["obs_date"].max()
add_check("V2", "V2.3", "Panel edges aligned across states",
          "PASS" if starts.nunique() == 1 and ends.nunique() == 1 else "WARN",
          metric=f"start {starts.min().date()}..{starts.max().date()}, end {ends.min().date()}..{ends.max().date()}",
          threshold="single common start and end",
          note="balanced monthly panel 1990-01 onward")

## V3 - Point-in-time & release calendar (information clock)

The counterpart to V2's observation clock. State CES is released in the *State Employment and Unemployment* report ~3 weeks after the national jobs report.

| Clock | Field | Convention | Validated by |
|---|---|---|---|
| **Observation time** - the month *measured* | `obs_date` | 1st of the reference month (pay period incl. 12th) | V2.1 |
| **Information time** - when it became *known* | `first_release_date` / vintage date | ~50 days after month end, Friday, joint state emp/unemp report | V3.2-V3.4 |

| Check | Question it answers |
|---|---|
| V3.1 | How far back can point-in-time be reconstructed? |
| V3.2 | Does the national vintage calendar match the BLS jobs-report schedule? |
| V3.3 | Is the state publication lag ~50 days as the joint report implies? |
| V3.4 | Do state first releases land after the national jobs report (~3 weeks later)? |
| V3.5 | Is the publication-lag regime stable over time? |
| V3.6 | Do months become available in order (monotone frontier)? |
| V3.7 | Does the PIT panel contain any look-ahead? |

In [ ]:
vint_start = df_v["first_release_date"].min()
add_check("V3", "V3.1", "ALFRED vintage boundary identified", "WARN",
          metric=f"first vintage {vint_start.date()}", threshold="documented",
          note="pre-boundary history exists only as current (fully-revised) values: usable for "
               "trend/seasonality estimation, NOT for point-in-time backtesting")

wd_names = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}
nat_wd = nat_vdates.dt.dayofweek.value_counts(normalize=True).sort_index()
fri_share = nat_wd.get(4, 0)
add_check("V3", "V3.2", "National PAYEMS vintages land on the jobs-report day (Friday)",
          "PASS" if fri_share >= 0.85 else "WARN",
          metric=", ".join(f"{wd_names[k]} {v:.0%}" for k, v in nat_wd.items() if v >= 0.01),
          threshold="Friday >= 85%",
          note="the national Employment Situation is released the first Friday after the reference month")

state_lag_med = df_v["pub_lag_days"].median()
add_check("V3", "V3.3", "State publication lag ~50 days (joint state emp/unemp report)",
          "PASS" if 40 <= state_lag_med <= 60 else "WARN",
          metric=f"median {state_lag_med:.0f}d (IQR {df_v['pub_lag_days'].quantile(.25):.0f}-{df_v['pub_lag_days'].quantile(.75):.0f})",
          threshold="median in [40, 60] days",
          note="~7 weeks after the reference month, i.e. ~3 weeks after the national jobs report")

# V3.4 state first release lands after the national first release for the same month
nat_rel = nat_v.set_index("obs_date")["first_release_date"]
al = df_v[["state", "obs_date", "first_release_date", "pub_lag_days"]].copy()
al["nat_release"] = al["obs_date"].map(nat_rel)
al = al.dropna(subset=["nat_release"])
al["gap_days"] = (al["first_release_date"] - al["nat_release"]).dt.days
after = (al["gap_days"] > 0).mean()
add_check("V3", "V3.4", "State NFP published after the national jobs report",
          "PASS" if after >= 0.95 else "WARN",
          metric=f"{after:.1%} of state months released after national; median gap +{al['gap_days'].median():.0f}d",
          threshold=">= 95% after national",
          note="state detail follows the national aggregate by ~3 weeks - a state feature is never available "
               "before the national print for the same month")

In [ ]:
# V3.5 lag-regime stability
df_v["rel_year"] = df_v["first_release_date"].dt.year
lag_by_year = df_v.groupby("rel_year")["pub_lag_days"].median()
lag_range = lag_by_year.max() - lag_by_year.min()
add_check("V3", "V3.5", "Publication-lag regime stable over time",
          "PASS" if lag_range <= 7 else "WARN",
          metric=f"median lag by year spans {lag_by_year.min():.0f}-{lag_by_year.max():.0f}d (range {lag_range:.0f}d)",
          threshold="yearly median range <= 7 days",
          note="a stable lag lets a backtest assume a single availability rule (unlike claims, which had a "
               "2020 capture-regime switch)")

# V3.6 availability-frontier monotonicity
inversions = 0
for state, g in df_v.groupby("state"):
    rel = g.sort_values("obs_date")["first_release_date"].values
    inversions += int((rel[1:] < rel[:-1]).sum())
add_check("V3", "V3.6", "Release dates weakly increasing in obs order (per state)",
          "PASS" if inversions == 0 else "WARN",
          metric=f"{inversions} inversions", threshold="== 0",
          note="the PIT builder uses the cummax release frontier, so small inversions are harmless")

# V3.7 PIT look-ahead guard
rel_lookup = df_v.set_index(["state", "obs_date"])["first_release_date"]
keys = pd.MultiIndex.from_arrays([pit["state"], pit["nfp_latest_obs"]])
pit_rel = rel_lookup.reindex(keys).values
violations = int((pd.to_datetime(pit_rel) > pit["as_of_date"]).sum())
unmatched = int(pd.isna(pit_rel).sum())
add_check("V3", "V3.7", "PIT panel look-ahead guard (nfp_latest_obs released <= as_of_date)",
          "PASS" if violations == 0 and unmatched == 0 else "FAIL",
          metric=f"{violations} violations, {unmatched} unmatched of {len(pit):,} rows", threshold="== 0")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(lag_by_year.index, lag_by_year.values, marker="o", color="#1f3864")
ax.axhspan(40, 60, color="steelblue", alpha=0.10)
ax.set(title="State NFP median publication lag by release year", ylabel="days after month end", xlabel="release year")
plt.tight_layout(); plt.show()

## V4 - Revision behavior (the central module for NFP)

Unlike claims, CES is revised almost every month, then **benchmarked annually** to QCEW. This module quantifies the size, the systematic direction, and the benchmark-month clustering - the evidence that PIT data is mandatory.

| Check | Question it answers |
|---|---|
| V4.1 | How often, and by how much, does a first print get revised? |
| V4.2 | Do revisions cluster at the annual benchmark (winter, with Jan data)? |
| V4.3 | Is there a systematic bias in first prints (do they over/understate)? |
| V4.4 | How large is the national benchmark itself (calibration)? |

In [ ]:
# V4.1 endpoint revision: first release vs current, all states
rev = df_v.merge(sa_c[["state", "obs_date", "value_current"]], on=["state", "obs_date"], how="inner")
rev["revision"] = rev["value_current"] - rev["value_first_release"]                 # thousands
rev["revision_pct"] = rev["revision"] / rev["value_first_release"] * 100
share_rev = (rev["revision"].abs() > 0.05).mean()
add_check("V4", "V4.1", "Endpoint revision rate & magnitude (first release vs current)",
          "WARN" if share_rev > 0.05 else "PASS",
          metric=f"{share_rev:.1%} of months revised; median |rev| {rev['revision_pct'].abs().median():.2f}%, "
                 f"p90 {rev['revision_pct'].abs().quantile(.9):.2f}%, max {rev['revision_pct'].abs().max():.1f}%",
          threshold="informational - NFP is heavily revised by design",
          note="the OPPOSITE of claims (~0% revised): current values embed benchmark corrections unknown in "
               "real time, so backtests MUST use first-release / vintage data")

# V4.3 directional bias of first prints
bias = rev["revision_pct"].mean()
add_check("V4", "V4.3", "Systematic bias in first prints",
          "PASS" if abs(bias) < 0.10 else "WARN",
          metric=f"mean revision {bias:+.3f}% (first prints {'understate' if bias>0 else 'overstate'} on average)",
          threshold="|mean revision| < 0.10% to treat as unbiased",
          note="a persistent sign means first prints are predictably off - a modellable nowcast signal, and a "
               "reason never to mix first-release features with current-data targets")

print(f"\nLargest upward revisions (first -> current, thousands of jobs):")
cols = ["state", "obs_date", "first_release_date", "value_first_release", "value_current", "revision", "revision_pct"]
top = rev.nlargest(8, "revision_pct")[cols].copy()
top["obs_date"] = top["obs_date"].dt.strftime("%Y-%m")
top["first_release_date"] = top["first_release_date"].dt.strftime("%Y-%m-%d")
print(top.to_string(index=False))
rev.to_csv("nfp_revisions.csv", index=False)
print("\nFull revision table saved to nfp_revisions.csv")

In [ ]:
# V4.2 benchmark clustering: full vintage trajectories (output_type=2) for sample states.
# For each observation month, find the RELEASE date of its single largest revision; the annual QCEW
# benchmark (published each winter with the January data) should make those cluster in Feb/Mar.
BENCH_SAMPLE = {"CANA": "California", "TXNA": "Texas", "NYNA": "New York"}
bench_months = []
for sid, name in BENCH_SAMPLE.items():
    d = fred_get({"series_id": sid, "realtime_start": "1776-07-04", "realtime_end": "9999-12-31",
                  "output_type": 2, "limit": 100000})
    t = pd.DataFrame(d["observations"]).set_index("date")
    vcols = [c for c in t.columns if c.startswith(sid)]
    vals = t[vcols].replace(".", np.nan).astype(float)
    vals.columns = pd.to_datetime([c.split("_", 1)[1] for c in vcols])
    for obs in vals.index:
        row = vals.loc[obs].dropna()
        if len(row) < 2:
            continue
        steps = row.diff().abs()
        if steps.max() > 0:
            bench_months.append(steps.idxmax().month)   # release month of the biggest revision
    time.sleep(0.3)
bm = pd.Series(bench_months).value_counts().sort_index()
winter_share = bm.reindex([1, 2, 3]).sum() / bm.sum()
add_check("V4", "V4.2", "Revisions cluster at the annual benchmark (Feb/Mar release months)",
          "PASS" if winter_share >= 0.6 else "WARN",
          metric=f"{winter_share:.0%} of largest revisions land in Jan-Mar; modal release month "
                 f"= {bm.idxmax()} ({bm.max()} of {bm.sum()} obs across CA/TX/NY)",
          threshold=">= 60% of biggest revisions in the winter benchmark window",
          note="confirms the annual QCEW re-anchoring drives the big revisions; a backtest must apply the "
               "benchmark on its release date, not spread it backward into the feature history")

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, ax = plt.subplots(figsize=(9.5, 3.2))
bars = ax.bar([month_names[m-1] for m in bm.index], bm.values,
              color=["#1f3864" if m in (1,2,3) else "steelblue" for m in bm.index])
ax.set(title="Release month of each obs-month's LARGEST revision (CA/TX/NY) - the annual benchmark",
       ylabel="count of obs months")
plt.tight_layout(); plt.show()

In [ ]:
# V4.4 national benchmark magnitude (PAYEMS advance vs fully-revised) for calibration
nat_rev = nat_v.merge(nat_cur[["obs_date", "value"]].rename(columns={"value": "current"}), on="obs_date", how="inner")
nat_rev["rev_pct"] = (nat_rev["current"] - nat_rev["value"]) / nat_rev["value"] * 100
add_check("V4", "V4.4", "National PAYEMS revision benchmark quantified",
          "PASS",
          metric=f"median {nat_rev['rev_pct'].median():+.2f}%, IQR [{nat_rev['rev_pct'].quantile(.25):+.2f}%, "
                 f"{nat_rev['rev_pct'].quantile(.75):+.2f}%], max |{nat_rev['rev_pct'].abs().max():.1f}%|",
          threshold="informational calibration",
          note="the national series is revised on the same schedule; its magnitude anchors what to expect "
               "state-by-state")

# ── visual 1: revision distribution ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.hist(rev["revision_pct"].clip(-3, 3), bins=70, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", ls="--", lw=1)
ax.axvline(rev["revision_pct"].median(), color="#1f3864", lw=1.5, label=f"median {rev['revision_pct'].median():+.2f}%")
ax.set(title="NFP first release -> current revision (% of first print, clipped +-3%)", xlabel="revision (%)", ylabel="count")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# ── visual 2: the revision wedge for a large state (first-release vintage vs current) ──
st = "California"
v_st = df_v[df_v["state"] == st].sort_values("obs_date")
c_st = sa_c[sa_c["state"] == st].sort_values("obs_date")
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(c_st["obs_date"], c_st["value_current"], lw=1.6, color="#1f3864", label="current (fully revised)")
ax.plot(v_st["obs_date"], v_st["value_first_release"], lw=1.1, color="darkorange", label="first release (real-time)")
ax.set(title=f"{st}: first-release vs current total nonfarm - the revision wedge", ylabel="employees (thousands)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## V5 - Outliers, breaks & anomalies

| Check | Question it answers |
|---|---|
| V5.1 | Which extreme monthly moves are real (COVID) vs unexplained? |
| V5.2 | Are there stale/frozen stretches (repeated identical levels)? |
| V5.3 | Are there level breaks beyond the COVID shock? |

In [ ]:
wide = sa_c.pivot(index="obs_date", columns="state", values="value_current").sort_index()
dlog = np.log(wide).diff()

# V5.1 extreme monthly moves, robust z (MAD)
out_rows = []
for state in wide.columns:
    d = dlog[state].dropna()
    mad = (d - d.median()).abs().median() * 1.4826
    z = (d - d.median()) / mad
    ext = z[z.abs() > 6]
    for dt, zz in ext.items():
        out_rows.append({"state": state, "obs_date": dt, "dlog": d[dt], "rz": zz})
outliers = pd.DataFrame(out_rows)
outliers["covid"] = outliers["obs_date"].between("2020-03-01", "2020-08-01")
unexplained = outliers[~outliers["covid"]]
add_check("V5", "V5.1", "Extreme monthly moves classified (|robust z| > 6 on dlog)",
          "PASS" if len(unexplained) / (wide.shape[0]*wide.shape[1]) < 0.002 else "WARN",
          metric=f"{len(outliers)} extreme moves: {int(outliers['covid'].sum())} COVID, {len(unexplained)} other",
          threshold="non-COVID extremes < 0.2% of panel",
          note="the 2020-03/04 collapse and rebound dominate; classify and regime-flag rather than winsorize away")
if len(unexplained):
    print(unexplained.nlargest(10, "rz", keep="all")[["state","obs_date","dlog","rz"]].to_string(index=False))

# V5.2 stale runs (identical consecutive levels - implausible for a state aggregate)
stale = []
for state, g in sa_c.groupby("state"):
    g = g.sort_values("obs_date")
    run = (g["value_current"] != g["value_current"].shift()).cumsum()
    for _, r in g.groupby(run):
        if len(r) >= 3:
            stale.append({"state": state, "start": r["obs_date"].min().date(), "months": len(r)})
add_check("V5", "V5.2", "No long stale-value runs (>= 3 identical months)",
          "PASS" if not stale else "WARN",
          metric=f"{len(stale)} runs", threshold="== 0",
          note="SA employment levels almost never repeat exactly; a run would signal a data hold")

# V5.3 level breaks beyond COVID (rolling 12m mean-shift on dlog, excluding 2020)
noncovid = dlog[(dlog.index < "2020-02-01") | (dlog.index > "2020-09-01")]
big_moves = (noncovid.abs() > 0.02).sum().sum()
add_check("V5", "V5.3", "Few extreme monthly moves outside COVID",
          "PASS",
          metric=f"{int(big_moves)} state-months with |MoM dlog| > 2% outside 2020",
          threshold="informational",
          note="monthly payroll growth is normally < 1%; large non-COVID moves flag small states or data events")

In [ ]:
# ── visual: extreme moves over time, COVID vs other ───────────────────────────
fig, ax = plt.subplots(figsize=(11, 3.2))
for lab, m, c in [("COVID", outliers["covid"], "crimson"), ("other", ~outliers["covid"], "purple")]:
    sub = outliers[m]
    ax.scatter(sub["obs_date"], sub["rz"].clip(-40, 40), s=12, alpha=0.5, color=c, label=f"{lab} ({len(sub)})")
ax.axhline(0, color="gray", lw=0.6)
ax.set(title="Extreme monthly payroll moves (|robust z| > 6)", ylabel="robust z (clipped +-40)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## V6 - Cross-sectional coherence

| Check | Question it answers |
|---|---|
| V6.1 | Do the 51 states sum to the national total (PAYEMS)? |
| V6.2 | Does every state co-move with the national payroll cycle? |

In [ ]:
state_sum = sa_c.groupby("obs_date")["value_current"].agg(["sum", "count"])
full = state_sum[state_sum["count"] == 51]
recon = full.join(nat_cur.set_index("obs_date")["value"].rename("national"), how="inner")
recon["ratio"] = recon["sum"] / recon["national"]
med_ratio = recon["ratio"].median()
add_check("V6", "V6.1", "Sum(51 states) reconciles with national PAYEMS",
          "PASS" if abs(med_ratio - 1) < 0.01 and recon["ratio"].std() < 0.01 else "WARN",
          metric=f"median ratio {med_ratio:.4f}, range [{recon['ratio'].min():.4f}, {recon['ratio'].max():.4f}]",
          threshold="median within 1% of 1.0, tight spread",
          note="state CES sums almost exactly to national (both SA total nonfarm) - a much tighter identity "
               "than claims, whose national excludes PR/VI")

yoy = np.log(wide).diff(12).dropna(how="all")
cmat = yoy.corr()
mean_corr = cmat.where(~np.eye(len(cmat), dtype=bool)).mean()
low = mean_corr[mean_corr < 0.30]
add_check("V6", "V6.2", "Cross-state co-movement (mean pairwise corr of YoY growth)",
          "PASS" if len(low) == 0 else "WARN",
          metric=f"panel median {mean_corr.median():.2f}; " + (f"low: {dict(low.round(2))}" if len(low) else "none < 0.30"),
          threshold="every state's mean corr >= 0.30",
          note="payroll cycles are highly synchronised across states; a decorrelated state is a data suspect")

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(recon.index, recon["ratio"], lw=0.9, color="#1f3864")
ax.axhline(1.0, color="red", ls="--", lw=1)
ax.set(title="Sum(51 states) / national PAYEMS", ylabel="ratio"); plt.tight_layout(); plt.show()

## V7 - Seasonality & stationarity (forecasting-critical)

The QC **flip** vs claims: NFP here is **seasonally adjusted**, so the test is that seasonality is *absent* - residual seasonality would be a defect. Levels **trend** (I(1) with drift), so work in growth rates.

| Check | Question it answers |
|---|---|
| V7.1 | Did seasonal adjustment actually remove the seasonality? (SA vs NSA) |
| V7.2 | Are the levels non-stationary (trending), and growth rates stationary? |
| V7.3 | Is payroll growth persistent (momentum) as expected? |

In [ ]:
# V7.1 residual seasonality: SA should show far weaker month-of-year structure than NSA.
# Metric: R^2 of month-dummies on MoM growth, SA vs NSA, averaged over sample states.
SAMPLE = ["California", "Texas", "New York", "Florida", "Ohio"]
wide_nsa = nsa_c.pivot(index="obs_date", columns="state", values="value_current").sort_index()

def seasonal_r2(series):
    g = np.log(series.dropna()).diff().dropna()
    g = g[(g.index < "2020-01-01")]                     # pre-COVID to avoid the shock
    m = pd.get_dummies(g.index.month).values.astype(float)
    yhat = m @ np.linalg.lstsq(m, g.values, rcond=None)[0]
    ss_res = ((g.values - yhat) ** 2).sum(); ss_tot = ((g.values - g.values.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot

r2_sa  = np.mean([seasonal_r2(wide[s])     for s in SAMPLE])
r2_nsa = np.mean([seasonal_r2(wide_nsa[s]) for s in SAMPLE])
add_check("V7", "V7.1", "Seasonal adjustment removed the seasonality (SA vs NSA month-dummy R2)",
          "PASS" if r2_sa < 0.15 and r2_sa < 0.3 * r2_nsa else "WARN",
          metric=f"month-of-year R2 of MoM growth: SA {r2_sa:.2f} vs NSA {r2_nsa:.2f}",
          threshold="SA R2 < 0.15 and < 30% of NSA",
          note="SA data should carry little month-of-year structure; strong residual seasonality would mean "
               "the published adjustment is inadequate for the modelling frequency")

# V7.2 stationarity: log level (expect non-stationary, trending) vs MoM growth (expect stationary)
from statsmodels.tsa.stattools import adfuller
rows = []
for s in SAMPLE:
    lvl = np.log(wide[s].dropna())
    p_lvl = adfuller(lvl, autolag="AIC")[1]
    p_g = adfuller(lvl.diff().dropna(), autolag="AIC")[1]
    rows.append({"state": s, "p_loglevel": p_lvl, "p_growth": p_g})
adf = pd.DataFrame(rows)
growth_ok = (adf["p_growth"] < 0.05).all()
level_trends = (adf["p_loglevel"] > 0.05).mean()
add_check("V7", "V7.2", "Transform recommendation (levels I(1) trending; growth stationary)",
          "PASS" if growth_ok else "WARN",
          metric="; ".join(f"{r.state}: lvl p={r.p_loglevel:.2f}, growth p={r.p_growth:.3f}" for r in adf.itertuples()),
          threshold="MoM growth stationary (p<0.05) for all sample states",
          note="model payroll GROWTH (MoM or YoY log-diff), never levels; levels are trending and non-stationary "
               "- the key contrast with cyclical claims/unemployment")

# V7.3 momentum: AR(1) of MoM growth (payroll growth is persistent)
ar1 = np.log(wide).diff().apply(lambda s: s.dropna()[s.dropna().index < '2020-01-01'].autocorr(1))
add_check("V7", "V7.3", "Payroll growth is persistent (AR1 of MoM growth, pre-COVID)",
          "PASS" if ar1.median() > 0.1 else "WARN",
          metric=f"median AR1 {ar1.median():.2f}, range [{ar1.min():.2f}, {ar1.max():.2f}]",
          threshold="median AR1 > 0.1 (positive momentum)",
          note="positive persistence supports momentum features; near-zero would imply monthly growth is noise")

In [ ]:
# ── visual: SA vs NSA month-of-year profile for one state (seasonality removed) ──
st = "California"
g_sa  = np.log(wide[st]).diff().dropna();  g_sa  = g_sa[g_sa.index < "2020-01-01"]
g_nsa = np.log(wide_nsa[st]).diff().dropna(); g_nsa = g_nsa[g_nsa.index < "2020-01-01"]
prof_sa  = g_sa.groupby(g_sa.index.month).mean() * 100
prof_nsa = g_nsa.groupby(g_nsa.index.month).mean() * 100
fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.plot(prof_nsa.index, prof_nsa.values, marker="o", color="darkorange", label="NSA (raw)")
ax.plot(prof_sa.index, prof_sa.values, marker="o", color="#1f3864", label="SA (adjusted)")
ax.axhline(0, color="gray", lw=0.6)
ax.set(title=f"{st}: mean MoM growth by calendar month, SA vs NSA (pre-2020)",
       xlabel="month", ylabel="mean MoM growth (%)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## V8 - Lead-lag readiness

| Check | Question it answers |
|---|---|
| V8.1 | Are the PIT features indexed by information time? |
| V8.2 | Is orientation correct - does payroll growth move opposite to unemployment (Okun)? |
| V8.3 | Alignment convention for pairing with monthly indicators. |

In [ ]:
monthly_asof = (pit["as_of_date"].dt.day == 1).all()
lag_med = pit["nfp_lag_months"].median()
add_check("V8", "V8.1", "PIT features indexed by information time (monthly as-of grid)",
          "PASS" if monthly_asof and 1 <= lag_med <= 3 else "WARN",
          metric=f"monthly as-of grid: {monthly_asof}; median data-staleness {lag_med:.0f} months",
          threshold="monthly grid; staleness 1-3 months (the ~50-day lag)",
          note="at as-of month T the latest available reference month is ~T-2 (the publication lag)")

# V8.2 orientation: payroll growth should move OPPOSITE to unemployment change (fetch UR)
ABBR = {name: ab for name, ab, _ in STATES}
ur_rows = []
for s in SAMPLE:
    for o in fred_obs({"series_id": f"{ABBR[s]}UR", "observation_start": "1990-01-01", "frequency": "m"}):
        ur_rows.append({"state": s, "date": pd.to_datetime(o["date"]), "ur": float(o["value"])})
    time.sleep(0.3)
ur = pd.DataFrame(ur_rows)
signs = []
for s in SAMPLE:
    g = np.log(wide[s]).diff(12).dropna()                       # YoY payroll growth
    du = ur[ur["state"] == s].set_index("date")["ur"].diff(12).dropna()  # YoY change in UR
    j = g.index.intersection(du.index)
    signs.append({"state": s, "corr": g.loc[j].corr(du.loc[j])})
sdf = pd.DataFrame(signs)
n_neg = int((sdf["corr"] < -0.3).sum())
add_check("V8", "V8.2", "Orientation: payroll growth moves opposite to unemployment (Okun sign)",
          "PASS" if n_neg >= 4 else "FAIL",
          metric="; ".join(f"{r.state}: corr {r.corr:+.2f}" for r in sdf.itertuples()),
          threshold=">= 4/5 states with corr < -0.3",
          note="validates sign/alignment wiring: more jobs <-> lower unemployment. NOT the analysis itself")

add_check("V8", "V8.3", "Monthly alignment convention documented", "WARN",
          metric="both NFP and LAUS reference the week/pay-period incl. the 12th",
          threshold="documented",
          note="NFP (payroll) and LAUS (household) share the 12th-of-month reference, so monthly joins align "
               "directly - but respect the ~3-week-longer NFP-state publication lag for real-time work")

## V9 - Backtest readiness

| Check | Question it answers |
|---|---|
| V9.1 | From which date is the PIT panel complete? |
| V9.2 | Is the universe stable (no state entry/exit)? |
| V9.3 | **How badly would using current data instead of vintage contaminate a backtest?** |

In [ ]:
cov = pit.groupby("as_of_date")["state"].nunique()
full_start = cov[cov == 51].index.min()
holes = cov[(cov.index > full_start) & (cov < 51)]
add_check("V9", "V9.1", "PIT coverage complete after ramp-in",
          "PASS" if len(holes) == 0 else "FAIL",
          metric=f"51/51 from {full_start.date()}; {len(holes)} holes after", threshold="0 holes",
          note=f"usable backtest window starts {full_start.date()}")

weeks_per_state = sa_c.groupby("state")["obs_date"].nunique()
add_check("V9", "V9.2", "Universe stability (balanced panel)",
          "PASS" if weeks_per_state.nunique() == 1 else "WARN",
          metric=f"months per state: {weeks_per_state.min()}-{weeks_per_state.max()}", threshold="identical")

# V9.3 the money check: current-vs-vintage contamination a naive backtest would suffer.
# For each PIT row, compare the first-release latest value to what CURRENT data shows for that month.
cur_lookup = sa_c.set_index(["state", "obs_date"])["value_current"]
pit_keys = pd.MultiIndex.from_arrays([pit["state"], pit["nfp_latest_obs"]])
cur_at_latest = cur_lookup.reindex(pit_keys).values
contamination = (cur_at_latest - pit["nfp_t0"].values) / pit["nfp_t0"].values * 100
contamination = pd.Series(contamination).dropna()
add_check("V9", "V9.3", "Current-vs-vintage contamination if PIT were ignored",
          "WARN" if contamination.abs().median() > 0.05 else "PASS",
          metric=f"median |gap| {contamination.abs().median():.2f}%, p90 {contamination.abs().quantile(.9):.2f}%, "
                 f"max {contamination.abs().max():.1f}% between current and real-time latest value",
          threshold="informational - drives the PIT-mandatory verdict",
          note="a backtest using current data would train on values the modeller could NOT have seen; unlike "
               "claims (~0% gap), NFP demands vintage features. This is the central backtesting finding.")

fig, ax = plt.subplots(figsize=(11, 2.6))
ax.plot(cov.index, cov.values, lw=1, color="#1f3864")
ax.set(title="States available per as-of month (PIT panel)", ylabel="# states"); plt.tight_layout(); plt.show()

## Extended visual diagnostics

Full plot set for the checks above - release timing, revision structure, cross-section and dynamics - so every result is visual as well as tabular.

In [ ]:
# ── Release & timing ─────────────────────────────────────────────────────────
_wdn = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}

# publication-lag distribution
fig, ax = plt.subplots(figsize=(9.5, 3))
ax.hist(df_v["pub_lag_days"], bins=range(int(df_v["pub_lag_days"].min()), int(df_v["pub_lag_days"].max())+2),
        color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(df_v["pub_lag_days"].median(), color="red", ls="--", lw=1,
           label=f"median {df_v['pub_lag_days'].median():.0f}d")
ax.set(title="State NFP publication-lag distribution", xlabel="days from reference month to first release", ylabel="count")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# state release date minus national jobs-report date
_natrel = nat_v.set_index("obs_date")["first_release_date"]
_al = df_v[["state","obs_date","first_release_date"]].copy()
_al["gap"] = (_al["first_release_date"] - _al["obs_date"].map(_natrel)).dt.days
_al = _al.dropna()
fig, ax = plt.subplots(figsize=(9.5, 3))
ax.hist(_al["gap"].clip(0, 40), bins=41, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(_al["gap"].median(), color="red", ls="--", lw=1, label=f"median +{_al['gap'].median():.0f}d after national")
ax.set(title="State NFP release minus national jobs-report date", xlabel="days after national release", ylabel="count")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# capture weekday
_wd = df_v["first_release_date"].dt.dayofweek.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar([_wdn[k] for k in _wd.index], _wd.values, color="steelblue")
ax.set(title="State NFP first-release capture weekday", ylabel="count"); plt.tight_layout(); plt.show()

In [ ]:
# ── Revision structure ───────────────────────────────────────────────────────
# revision by state (top revisers by mean |rev%|)
_bs = rev.groupby("state")["revision_pct"].apply(lambda s: s.abs().mean()).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(_bs.index)[::-1], list(_bs.values)[::-1], color="steelblue")
ax.set(title="Mean absolute revision by state (top 15)", xlabel="mean |revision| (% of first print)")
plt.tight_layout(); plt.show()

# revision magnitude over time
fig, ax = plt.subplots(figsize=(10.5, 3.2))
ax.scatter(rev["obs_date"], rev["revision_pct"].clip(-5, 5), s=7, alpha=0.3, color="steelblue")
ax.axhline(0, color="gray", lw=0.8)
ax.set(title="NFP revision size over time (clipped +-5%)", ylabel="current vs first (%)")
plt.tight_layout(); plt.show()

# national PAYEMS revision distribution
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(nat_rev["rev_pct"].clip(-1.5, 1.5), bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", ls="--", lw=1)
ax.set(title="National PAYEMS first-release -> current revision", xlabel="revision (%)", ylabel="count")
plt.tight_layout(); plt.show()

In [ ]:
# ── Cross-section ────────────────────────────────────────────────────────────
# all-states payroll levels (log spaghetti) - trending
fig, ax = plt.subplots(figsize=(11, 3.6))
for c in wide.columns:
    ax.plot(wide.index, wide[c], lw=0.5, alpha=0.4, color="steelblue")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set(title="All states: total nonfarm payroll levels (log scale) - trending, not cyclical",
       ylabel="employees (thousands)")
plt.tight_layout(); plt.show()

# cross-state co-movement bars
_cm = np.log(wide).diff(12).corr()
_cm = _cm.where(~np.eye(len(_cm), dtype=bool)).mean().sort_values()
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(len(_cm)), _cm.values, color="steelblue")
ax.set_xticks(range(len(_cm))); ax.set_xticklabels(_cm.index, rotation=90, fontsize=6)
ax.axhline(0.30, color="red", ls="--", lw=1, label="0.30 floor")
ax.set(title="Cross-state co-movement: mean pairwise corr of YoY payroll growth", ylim=(0, 1))
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# ── Dynamics & backtest ──────────────────────────────────────────────────────
# growth persistence bars (AR1 of MoM growth, pre-2020)
_ar1 = np.log(wide).diff().apply(lambda s: s.dropna()[s.dropna().index < "2020-01-01"].autocorr(1)).sort_values()
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(len(_ar1)), _ar1.values, color="steelblue")
ax.set_xticks(range(len(_ar1))); ax.set_xticklabels(_ar1.index, rotation=90, fontsize=6)
ax.axhline(0, color="gray", lw=0.6)
ax.set(title="Payroll growth persistence: AR(1) of MoM growth by state (pre-2020)")
plt.tight_layout(); plt.show()

# Okun cross-correlation: payroll growth (t) vs unemployment change (t+k)
_ks = list(range(-6, 7))
fig, ax = plt.subplots(figsize=(10, 3.4))
for s in SAMPLE:
    g = np.log(wide[s]).diff(12)
    du = ur[ur["state"] == s].set_index("date")["ur"].diff(12)
    prof = []
    for k in _ks:
        duk = du.shift(-k)
        j = g.index.intersection(duk.dropna().index)
        prof.append(g.reindex(j).corr(duk.reindex(j)))
    ax.plot(_ks, prof, marker=".", lw=1.2, label=s)
ax.axvline(0, color="gray", ls=":", lw=0.8); ax.axhline(0, color="gray", lw=0.8)
ax.set(title="Okun cross-correlation: payroll growth (t) vs unemployment change (t+k)",
       xlabel="k (months)", ylabel="corr")
ax.legend(fontsize=8, ncol=5); plt.tight_layout(); plt.show()

# feature contamination by as-of year (current vs vintage latest value)
_cur = sa_c.set_index(["state", "obs_date"])["value_current"]
_keys = pd.MultiIndex.from_arrays([pit["state"], pit["nfp_latest_obs"]])
_gap = (_cur.reindex(_keys).values - pit["nfp_t0"].values) / pit["nfp_t0"].values * 100
_cont = pd.Series(np.abs(_gap), index=pit["as_of_date"].dt.year).groupby(level=0).median()
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.bar(_cont.index.astype(int), _cont.values, color="steelblue")
ax.set(title="Current-vs-vintage gap by as-of year (median |gap|, %)", ylabel="% of value")
plt.tight_layout(); plt.show()

## V10 - Reproducibility & governance

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "validated_at": pd.Timestamp.now().isoformat(),
    "source": "FRED/ALFRED - series {ABBR}NA (SA total nonfarm), {ABBR}NAN (NSA), PAYEMS",
    "vintage_params": {"realtime_start": "1776-07-04", "realtime_end": "9999-12-31", "output_type": 4},
    "files": {f: {"sha256": sha256(f), "rows": int(pd.read_csv(f).shape[0])} for f in DATA_FILES.values()},
    "data_ranges": {
        "current_history": [str(df_c["obs_date"].min().date()), str(df_c["obs_date"].max().date())],
        "vintage_window":  [str(df_v["first_release_date"].min().date()), str(df_v["first_release_date"].max().date())],
        "pit_panel":       [str(pit["as_of_date"].min().date()), str(pit["as_of_date"].max().date())],
    },
}
with open("nfp_validation_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
add_check("V10", "V10.1", "Manifest written (hashes, params, ranges)", "PASS",
          metric="nfp_validation_manifest.json", threshold="file exists",
          note="ALFRED queries with pinned realtime params are reproducible; re-fetch and diff hashes to audit")

## Validation report

In [ ]:
QUESTIONS = {
    "V1.1":"Do we have all 51 states in every table?", "V1.2":"Is (state, adjustment, month) a unique key?",
    "V1.3":"Are key fields complete (no nulls)?", "V1.4":"Is every payroll level strictly positive and finite?",
    "V1.5":"Are the series lengths consistent across states?",
    "V2.1":"Are all observation dates month-starts?", "V2.2":"Are there missing months in any state?",
    "V2.3":"Do all states cover the same window?",
    "V3.1":"How far back can point-in-time be reconstructed?", "V3.2":"Does the vintage calendar match the jobs-report schedule?",
    "V3.3":"Is the state publication lag ~50 days?", "V3.4":"Do state releases follow the national jobs report?",
    "V3.5":"Is the publication-lag regime stable over time?", "V3.6":"Do months become available in order?",
    "V3.7":"Does the PIT panel contain any look-ahead?",
    "V4.1":"How often/how much does a first print get revised?", "V4.2":"Do revisions cluster at the annual benchmark?",
    "V4.3":"Is there a systematic bias in first prints?", "V4.4":"How large is the national benchmark (calibration)?",
    "V5.1":"Which extreme moves are real (COVID) vs unexplained?", "V5.2":"Are there stale/frozen stretches?",
    "V5.3":"Are there level breaks beyond COVID?",
    "V6.1":"Do the 51 states sum to national PAYEMS?", "V6.2":"Does every state co-move with the national cycle?",
    "V7.1":"Did SA actually remove the seasonality?", "V7.2":"Are levels trending and growth stationary?",
    "V7.3":"Is payroll growth persistent?",
    "V8.1":"Are features indexed by information time?", "V8.2":"Does payroll move opposite to unemployment (Okun)?",
    "V8.3":"How to align with monthly indicators?",
    "V9.1":"From which date is the PIT panel complete?", "V9.2":"Is the universe stable?",
    "V9.3":"How badly would current-instead-of-vintage contaminate a backtest?",
    "V10.1":"Can the dataset be reproduced and audited?",
}

report = pd.DataFrame(checks)[["module", "id", "check", "result", "metric", "threshold", "note"]]
report.insert(3, "question", report["id"].map(QUESTIONS).fillna(""))
report.to_csv("nfp_validation_report.csv", index=False)

counts = report["result"].value_counts()
print(f"Checks: {len(report)}  |  PASS {counts.get('PASS',0)}  WARN {counts.get('WARN',0)}  FAIL {counts.get('FAIL',0)}\n")
missing_q = report.loc[report["question"] == "", "id"].tolist()
if missing_q:
    print(f"NOTE: checks without a mapped question: {missing_q}")

def color(v):
    return {"PASS": "background-color:#d4edda", "WARN": "background-color:#fff3cd",
            "FAIL": "background-color:#f8d7da"}.get(v, "")
report.style.map(color, subset=["result"]).hide(axis="index")

## Gating decisions (go/no-go)

In [ ]:
n_fail = int((report["result"] == "FAIL").sum())
print("=" * 78)
print(f"GATE STATUS: {'BLOCKED - resolve FAILs first' if n_fail else 'CLEARED with documented caveats'}")
print("=" * 78)
print(f"""
G1 - SCOPE CLEARED
  * Backtesting / PIT work: {full_start.date()} onward (51/51 PIT coverage; strict ALFRED vintages)
  * Trend / seasonality estimation: full 1990+ current history
  * NOT cleared: treating pre-{vint_start.date()} data as point-in-time

G2 - REQUIRED TRANSFORMS (from V7)
  * Model payroll GROWTH (MoM or YoY log-diff) - levels are I(1) trending, never stationary
  * Use the SA series (residual seasonality verified small in V7.1); reach for NSA only with own adjustment
  * Regime-flag 2020 (COVID collapse/rebound dwarfs every other move)

G3 - POINT-IN-TIME IS MANDATORY (the headline, from V4/V9)
  * ~{share_rev:.0%} of months are revised; current data embeds annual QCEW benchmarks unknown in real time
  * Backtests MUST use first-release / vintage features (nfp_pit_features.csv); a naive current-data backtest
    carries a median {contamination.abs().median():.2f}% (p90 {contamination.abs().quantile(.9):.2f}%) look-ahead gap per state-month
  * Apply benchmark revisions on their release date; never spread them backward into the feature history
  * National PAYEMS revision (median {nat_rev['rev_pct'].median():+.2f}%) calibrates the expected state benchmark size
""")